# SE3 Imbalance Price Forecasting

End-to-end notebook for forecasting SE3 15-minute imbalance prices using
LightGBM quantile regression.  No DuckDB or environment variables required
to run the data fetch.

**Sections:**
1. Fetch Data — eSett imbalance + Open-Meteo weather + ENTSOE spot (optional)
2. EDA — time series, distributions, temporal patterns, autocorrelation, wind
3. Feature Engineering — short lags, direction, rolling stats, calendar, weather
4. Baselines — persistence, rolling mean, spot proxy
5. Train — LightGBM p05 / p50 / p95 quantile models
6. Evaluate — metrics, plots, feature importance, multi-horizon comparison
7. Summary — key results and next steps

## Data Transformations — Quick Reference

This notebook applies the following transformations at each pipeline stage.
Detailed notes appear again inside each section; this table is the at-a-glance
index.

### 1. Fetch & Ingest

| Step | Transformation |
|---|---|
| eSett API params | `date` → ISO-8601 + milliseconds + `Z` suffix (`2025-01-01T00:00:00.000Z`) |
| eSett timestamps | UTC string → `pd.Timestamp` → `tz_convert("Europe/Stockholm")` |
| `mainDirRegPowerPerMBA` | float or string → `np.sign()` → integer {−1, 0, +1} |
| Open-Meteo request | `timezone="UTC"` avoids DST ambiguity at clock-change boundaries |
| Open-Meteo timestamps | UTC strings → `tz_localize("UTC")` → `tz_convert("Europe/Stockholm")` |
| Weather upsampling | Hourly → 15-min via `reindex` + `ffill` |
| Spot prices (optional) | Hourly ENTSOE → 15-min via `reindex` + `ffill`; skipped if no API key |
| Merge | Left-join imbalance ← weather ← spot on shared 15-min index |

### 2. Feature Engineering (`build_imbalance_features`)

| Group | Transformation | Leakage guard |
|---|---|---|
| **Price lags** | `imbl_price.shift(n)` for n = 1, 2, 4, 8, 16 periods + 1 h / 2 h / 4 h | shift avoids lookahead |
| **Direction lags** | `direction.shift(n)` for n = 1, 2, 4, 8 | shift avoids lookahead |
| **Direction streak** | Vectorised run-length: `(d ≠ d.shift(1)).cumsum()` → `groupby.cumcount()+1` on `d.shift(1)` | uses lagged direction |
| **Rolling stats** | `p.shift(1).rolling(4/16/96).mean/std` | shifted 1 period first |
| **EWA** | `p.shift(1).ewm(span=8).mean()` | shifted 1 period first |
| **Regulation spread** | `(up_reg − down_reg).shift(1)` | guarded: skipped if source is all-NaN |
| **Spot diff lag** | `imbl_spot_diff.shift(1)` | guarded: skipped if source is all-NaN |
| **Spot price lag** | `spot_price.shift(1)` | guarded: skipped if no ENTSOE key |
| **Calendar** | `slot = hour×4 + minute÷15` (0–95); `sin/cos` for hour, dow, month, slot | purely deterministic |
| **Fourier harmonics** | k = 1, 2, 3 for daily/weekly (96-slot base); k = 1, 2 for annual | purely deterministic |
| **Binary flags** | `is_peak`, `is_night`, `is_weekend`, `is_holiday` | purely deterministic |
| **Season** | Month → {1 winter, 2 spring, 3 summer, 4 autumn} | purely deterministic |
| **Wind surprise** | `wind − wind.rolling(672).mean()` (7 d × 96 = 672 periods) | rolling on current (weather is observed) |
| **Wind interactions** | `wind × is_night`, `wind × is_peak`, `wind²` | derived from weather |
| **Cloudcover lag** | `cloudcover.shift(1)` | guarded: skipped if all-NaN |
| **Target** | `imbl_price.shift(−horizon)` — price at t+horizon | only created, not in feature set |

### 3. Pre-training Safety Net (baselines cell)

Three-step approach that prevents silent all-NaN failures:

```python
df_feats     = df_feats.dropna(axis=1, how="all")          # drop columns that are entirely NaN
feature_cols = make_imbalance_feature_cols(df_feats)        # rebuild list (filters notna().any())
df_model     = df_feats.dropna(subset=feature_cols + ["target"])  # drop rows with any remaining NaN
```

### 4. Train / Test Split

| Parameter | Value | Rationale |
|---|---|---|
| Test set | Last 60 days | Denser than the 90-day spot-price test (15-min vs hourly) |
| Validation set | Last 10 % of train | For LightGBM early stopping (`patience=50`) |
| No Ridge subtraction | — | Imbalance lacks the clean linear day-of-year structure of spot prices |

In [ ]:
from __future__ import annotations

import logging
import os
import warnings
from datetime import date, timedelta
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from statsmodels.graphics.tsaplots import plot_acf as sm_acf
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("imbalance")

# ── Constants ──────────────────────────────────────────────────────────────────
WEATHER_LAT:  float = 59.33       # Stockholm / central SE3
WEATHER_LON:  float = 18.07
SE3_MBA:      str   = "10Y1001A1001A46L"
FETCH_MONTHS: int   = 18
CHUNK_MONTHS: int   = 2
HORIZON:      int   = 1           # default forecast horizon (15-min periods)
TEST_DAYS:    int   = 60
FIGURES_DIR         = Path("figures/imbalance")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LGBM_PARAMS: dict = {
    "n_estimators":      2000,
    "learning_rate":     0.05,
    "num_leaves":        63,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "verbose":          -1,
}

SE_HOLIDAYS = holidays.Sweden()
ENTSOE_KEY  = os.environ.get("ENTSOE_API_KEY", "")

print(f"ENTSOE key available : {bool(ENTSOE_KEY)}")
print(f"statsmodels available: {HAS_STATSMODELS}")
print(f"LightGBM version     : {lgb.__version__}")

## 1. Fetch Data

Three sources merged to a 15-min aligned DataFrame.  Only the eSett imbalance fetch is required; weather and spot gracefully degrade to NaN if unavailable.

**Key encoding decisions in this section:**

- eSett timestamps arrive as CET-local tz-naive strings; we use `timestampUTC`
  (the parallel UTC field) and `tz_localize("UTC")` to sidestep DST ambiguity.
- Open-Meteo is requested with `timezone="UTC"` for the same reason — the
  Stockholm DST clock-back at `2025-10-26 02:00:00` would cause an
  `AmbiguousTimeError` if we localised a tz-naive local timestamp directly.
- Weather is hourly; imbalance is 15-min. We upsample weather with `ffill`
  (forward-fill) rather than interpolation, matching the assumption used in
  `pipeline.py sync_weather`.

### 1a. eSett Imbalance Prices

[eSett Open Data EXP14/Prices](https://api.opendata.esett.com/EXP14/Prices) —
free, no authentication.  Data is 15-min resolution in CET local time.

Fields used:
- `imblSalesPrice` → `imbl_price` (EUR/MWh)
- `mainDirRegPowerPerMBA` → `direction` (−1 long, 0 neutral, +1 short)
- `imblSpotDifferencePrice` → `imbl_spot_diff`
- `upRegPrice` / `downRegPrice` → regulation spread

We chunk requests by 2 months to avoid gateway timeouts.

In [ ]:
ESETT_URL = "https://api.opendata.esett.com/EXP14/Prices"


def _encode_direction(raw: pd.Series) -> pd.Series:
    """Map mainDirRegPowerPerMBA to -1 (long), 0 (neutral), +1 (short)."""
    if raw.dtype == object:
        dmap = {"LONG": -1, "NEUTRAL": 0, "SHORT": 1}
        return raw.str.upper().map(dmap).fillna(0).astype(int)
    return np.sign(pd.to_numeric(raw, errors="coerce").fillna(0)).astype(int)


def _fetch_imbalance_chunk(start: date, end: date) -> pd.DataFrame:
    """
    Fetch one eSett EXP14/Prices chunk for SE3.

    Parameters
    ----------
    start, end : date
        Inclusive date range (keep to ≤2 months to avoid timeouts).

    Returns
    -------
    pd.DataFrame
        15-min index (Europe/Stockholm) with columns:
        imbl_price, direction, imbl_spot_diff, up_reg_price, down_reg_price.
    """
    # API requires full ISO-8601 with milliseconds and Z suffix
    from datetime import datetime as _dt, timezone as _tz
    start_dt = _dt(start.year, start.month, start.day, 0, 0, 0, tzinfo=_tz.utc)
    end_dt   = _dt(end.year,   end.month,   end.day,   23, 59, 59, tzinfo=_tz.utc)
    params = {
        "start": start_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        "end":   end_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        "mba":   SE3_MBA,
    }
    r = requests.get(ESETT_URL, params=params, timeout=90)
    r.raise_for_status()
    data = r.json()

    records = data if isinstance(data, list) else data.get("data", data.get("prices", []))
    if not records:
        log.warning("Empty eSett response for %s → %s", start, end)
        return pd.DataFrame()

    df = pd.DataFrame(records)

    ts_col = next(
        (c for c in ["timestampUTC", "startTime", "deliveryStart", "time", "timestamp"]
         if c in df.columns),
        None,
    )
    if ts_col is None:
        log.warning("No timestamp column in: %s", df.columns.tolist())
        return pd.DataFrame()

    raw_ts = pd.to_datetime(df[ts_col])
    if ts_col == "timestampUTC":
        # Already UTC — convert directly
        df["timestamp"] = raw_ts.dt.tz_localize("UTC") if raw_ts.dt.tz is None \
            else raw_ts.dt.tz_convert("UTC")
        df["timestamp"] = df["timestamp"].dt.tz_convert("Europe/Stockholm")
    elif raw_ts.dt.tz is None:
        df["timestamp"] = raw_ts.dt.tz_localize(
            "Europe/Stockholm", ambiguous="infer", nonexistent="shift_forward"
        )
    else:
        df["timestamp"] = raw_ts.dt.tz_convert("Europe/Stockholm")

    rename = {
        "imblSalesPrice":          "imbl_price",
        "mainDirRegPowerPerMBA":   "direction_raw",
        "imblSpotDifferencePrice": "imbl_spot_diff",
        "upRegPrice":              "up_reg_price",
        "downRegPrice":            "down_reg_price",
    }
    df = df.rename(columns=rename).set_index("timestamp").sort_index()

    if "direction_raw" in df.columns:
        df["direction"] = _encode_direction(df["direction_raw"])
    else:
        df["direction"] = 0

    keep = [c for c in ["imbl_price", "direction", "imbl_spot_diff",
                         "up_reg_price", "down_reg_price"] if c in df.columns]
    return df[keep].apply(pd.to_numeric, errors="coerce")


def fetch_all_imbalance(months: int = FETCH_MONTHS) -> pd.DataFrame:
    """
    Fetch SE3 imbalance prices for the past `months` months.
    Chunks by CHUNK_MONTHS to avoid gateway timeouts.

    Returns
    -------
    pd.DataFrame
        15-min resolution, Europe/Stockholm, deduplicated and sorted.
    """
    end_dt   = date.today()
    start_dt = end_dt - timedelta(days=int(months * 30.5))
    chunks: list[pd.DataFrame] = []
    current = start_dt

    while current < end_dt:
        m2 = current.month + CHUNK_MONTHS
        y2 = current.year + (m2 - 1) // 12
        m2 = (m2 - 1) % 12 + 1
        chunk_end = min(date(y2, m2, 1) - timedelta(days=1), end_dt)

        log.info("Fetching imbalance %s → %s", current, chunk_end)
        try:
            chunk = _fetch_imbalance_chunk(current, chunk_end)
            if not chunk.empty:
                chunks.append(chunk)
        except requests.RequestException as exc:
            log.warning("eSett fetch failed %s→%s: %s", current, chunk_end, exc)
        current = chunk_end + timedelta(days=1)

    if not chunks:
        raise RuntimeError("No imbalance data fetched — check API connectivity.")

    df = pd.concat(chunks)
    df = df[~df.index.duplicated(keep="first")].sort_index()
    log.info(
        "Imbalance: %d rows  %s → %s",
        len(df), df.index.min().date(), df.index.max().date(),
    )
    return df

In [ ]:
df_imbl = fetch_all_imbalance(months=FETCH_MONTHS)
print(f"Shape : {df_imbl.shape}")
print(f"Period: {df_imbl.index.min().date()} → {df_imbl.index.max().date()}")
print(f"\nNull counts:")
print(df_imbl.isnull().sum())
df_imbl.head()

### 1b. Open-Meteo Weather (archive, no DuckDB)

Fetches the same variables as `pipeline.py sync_weather` but writes directly
to a DataFrame — no caching layer needed for exploratory work.  Hourly data
is upsampled to 15-min via forward fill.

In [ ]:
def fetch_weather(start: date, end: date) -> pd.DataFrame:
    """
    Fetch historical weather from Open-Meteo archive API.

    Parameters
    ----------
    start, end : date
        Date range.  Archive lags ~2 days so `end` is capped automatically.

    Returns
    -------
    pd.DataFrame
        15-min indexed (Europe/Stockholm) columns:
        temperature, windspeed_10m, windspeed_100m, solar_radiation, cloudcover.
    """
    end_capped = min(end, date.today() - timedelta(days=2))
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   WEATHER_LAT,
        "longitude":  WEATHER_LON,
        "start_date": start.isoformat(),
        "end_date":   end_capped.isoformat(),
        "hourly":     "temperature_2m,windspeed_10m,windspeed_100m,direct_radiation,cloudcover",
        "timezone":   "UTC",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    hourly = r.json().get("hourly", {})

    df = pd.DataFrame({
        "timestamp":       pd.to_datetime(hourly["time"]),
        "temperature":     hourly.get("temperature_2m"),
        "windspeed_10m":   hourly.get("windspeed_10m"),
        "windspeed_100m":  hourly.get("windspeed_100m"),
        "solar_radiation": hourly.get("direct_radiation"),
        "cloudcover":      hourly.get("cloudcover"),
    })
    # API returns UTC timestamps — localize as UTC then convert to Stockholm
    df["timestamp"] = (
        df["timestamp"].dt.tz_localize("UTC")
        .dt.tz_convert("Europe/Stockholm")
    )
    df = df.set_index("timestamp").sort_index()

    # Upsample hourly → 15-min with forward fill
    idx_15min = pd.date_range(
        df.index.min(), df.index.max(), freq="15min", tz="Europe/Stockholm"
    )
    return df.reindex(idx_15min).ffill()


weather_start = date.today() - timedelta(days=int(FETCH_MONTHS * 30.5))
weather_end   = date.today()
df_weather    = fetch_weather(weather_start, weather_end)

print(f"Shape : {df_weather.shape}")
print(f"Period: {df_weather.index.min().date()} → {df_weather.index.max().date()}")
df_weather.head()

### 1c. ENTSOE Spot Prices (optional)

Day-ahead SE3 spot prices are a useful feature (imbalance is often a spread
around spot).  We reuse `ENTSOE_API_KEY` from the environment if available;
otherwise the column is filled with NaN and the model runs without it.

In [ ]:
df_spot = pd.DataFrame(
    index=df_weather.index, columns=["spot_price"], dtype=float
)  # default: NaN column

if ENTSOE_KEY:
    try:
        from entsoe import EntsoePandasClient

        client    = EntsoePandasClient(api_key=ENTSOE_KEY)
        spot_s    = pd.Timestamp(weather_start.isoformat(), tz="Europe/Stockholm")
        spot_e    = pd.Timestamp(weather_end.isoformat(),   tz="Europe/Stockholm")
        raw_spot  = client.query_day_ahead_prices(SE3_MBA, start=spot_s, end=spot_e)
        raw_spot  = raw_spot.rename("spot_price").to_frame()
        raw_spot.index = raw_spot.index.tz_convert("Europe/Stockholm")
        raw_spot.index.name = "timestamp"

        # Upsample hourly → 15-min
        idx_15min = pd.date_range(
            raw_spot.index.min(), raw_spot.index.max(), freq="15min",
            tz="Europe/Stockholm",
        )
        df_spot = raw_spot.reindex(idx_15min).ffill()
        print(
            f"Spot prices: {len(df_spot):,} rows  "
            f"{df_spot.index.min().date()} → {df_spot.index.max().date()}"
        )
    except Exception as exc:
        log.warning("Spot price fetch failed: %s — continuing with NaN.", exc)
else:
    print("No ENTSOE_API_KEY — spot_price will be NaN (model works without it).")

### 1d. Merge & Align

Left-join imbalance (anchor) with weather and spot on the 15-min index.

In [ ]:
df_merged = (
    df_imbl
    .join(df_weather, how="left")
    .join(df_spot,    how="left")
    .sort_index()
)

print(f"Merged: {len(df_merged):,} rows  "
      f"{df_merged.index.min().date()} → {df_merged.index.max().date()}")
print(f"\nNull counts:")
print(df_merged.isnull().sum())
df_merged.head()

## 2. EDA

Thorough exploratory analysis before any modelling.  Key questions:

- What does the imbalance price distribution look like?
- Are there clear intraday / weekly / seasonal patterns?
- How persistent is the imbalance signal (autocorrelation)?
- How strongly does wind speed relate to imbalance price?

### 2a. Full Time Series (hourly median)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
hourly_med = df_merged["imbl_price"].resample("h").median()
ax.plot(hourly_med.index, hourly_med.values, linewidth=0.6, color="steelblue")
ax.axhline(0, color="red", linewidth=0.8, linestyle="--", alpha=0.7)
ax.set_xlabel("Date")
ax.set_ylabel("Imbalance price (EUR/MWh)")
ax.set_title("SE3 Imbalance Price — hourly median")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_timeseries.png", dpi=150)
plt.show()

### 2b. Price Distribution

In [ ]:
p = df_merged["imbl_price"].dropna()
percentiles = [1, 5, 25, 50, 75, 95, 99]
pct_vals = np.percentile(p, percentiles)

print("Percentile table (EUR/MWh):")
print(f"  {'p':>4}  {'value':>8}")
for pct, val in zip(percentiles, pct_vals):
    print(f"  {pct:>4}  {val:>8.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram (clipped for readability)
clip_lo, clip_hi = np.percentile(p, 1), np.percentile(p, 99)
p_clipped = p.clip(clip_lo, clip_hi)
axes[0].hist(p_clipped, bins=80, color="steelblue", edgecolor="none", alpha=0.8)
axes[0].axvline(0, color="red", linewidth=1, linestyle="--")
axes[0].set_xlabel("Imbalance price (EUR/MWh)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution (p1-p99 clipped)")

# CDF
sorted_p = np.sort(p)
axes[1].plot(sorted_p, np.linspace(0, 1, len(sorted_p)), color="steelblue")
for pct, val in zip([5, 50, 95], [pct_vals[1], pct_vals[3], pct_vals[5]]):
    axes[1].axvline(val, color="orange", linewidth=0.8, linestyle="--")
    axes[1].text(val, pct/100, f" p{pct}", fontsize=8, va="bottom")
axes[1].set_xlabel("Imbalance price (EUR/MWh)")
axes[1].set_ylabel("Cumulative probability")
axes[1].set_title("CDF")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_distribution.png", dpi=150)
plt.show()

### 2c. Direction Analysis

`direction` encodes the dominant imbalance direction each 15-min period:
−1 = long (surplus), 0 = neutral, +1 = short (deficit).

In [ ]:
dir_counts = df_merged["direction"].value_counts().sort_index()
dir_labels = {-1: "Long (−1)", 0: "Neutral (0)", 1: "Short (+1)"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of direction counts
bars = axes[0].bar(
    [dir_labels.get(k, str(k)) for k in dir_counts.index],
    dir_counts.values,
    color=["#3b82f6", "#6b7280", "#ef4444"],
)
axes[0].set_ylabel("Count")
axes[0].set_title("Direction distribution")
for bar, val in zip(bars, dir_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"{val/len(df_merged)*100:.1f}%", ha="center", fontsize=9)

# Boxplot of imbalance price by direction
groups = [
    df_merged.loc[df_merged["direction"] == d, "imbl_price"].dropna()
    for d in [-1, 0, 1]
]
bp = axes[1].boxplot(
    groups, labels=[dir_labels[-1], dir_labels[0], dir_labels[1]],
    patch_artist=True, showfliers=False,
)
for patch, color in zip(bp["boxes"], ["#3b82f6", "#6b7280", "#ef4444"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_ylabel("Imbalance price (EUR/MWh)")
axes[1].set_title("Imbalance price by direction")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_direction.png", dpi=150)
plt.show()

### 2d. Temporal Patterns (intraday, day-of-week, monthly)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Intraday pattern: median by hour
hourly_pat = df_merged.copy()
hourly_pat["hour"] = hourly_pat.index.hour
intraday = hourly_pat.groupby("hour")["imbl_price"].median()
axes[0].bar(intraday.index, intraday.values, color="steelblue", alpha=0.8)
axes[0].axhline(0, color="red", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Hour of day (CET)")
axes[0].set_ylabel("Median imbalance price (EUR/MWh)")
axes[0].set_title("Intraday pattern")
axes[0].set_xticks(range(0, 24, 3))

# Day-of-week pattern
dow_pat = df_merged.copy()
dow_pat["dow"] = dow_pat.index.dayofweek
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_med = dow_pat.groupby("dow")["imbl_price"].median()
axes[1].bar(dow_labels, dow_med.values,
            color=["steelblue"]*5 + ["#f59e0b"]*2, alpha=0.8)
axes[1].axhline(0, color="red", linewidth=0.8, linestyle="--")
axes[1].set_ylabel("Median imbalance price (EUR/MWh)")
axes[1].set_title("Day-of-week pattern")

# Monthly trend
monthly = df_merged["imbl_price"].resample("ME").median()
axes[2].plot(monthly.index, monthly.values, marker="o", markersize=3,
             color="steelblue", linewidth=1.2)
axes[2].axhline(0, color="red", linewidth=0.8, linestyle="--")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Median imbalance price (EUR/MWh)")
axes[2].set_title("Monthly trend")
fig.autofmt_xdate(rotation=30)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_temporal_patterns.png", dpi=150)
plt.show()

### 2e. Autocorrelation (up to 12 h = 48 lags at 15-min)

High autocorrelation at short lags means recent periods are the strongest
predictors — this motivates our short-lag feature set.

In [ ]:
MAX_LAGS = 48  # 12 h at 15-min resolution


def _manual_acf(series: pd.Series, max_lags: int = MAX_LAGS) -> np.ndarray:
    """Autocorrelation function without statsmodels dependency."""
    s = series.dropna().values.astype(float)
    n = len(s)
    m = s.mean()
    v = ((s - m) ** 2).mean()
    acf = [1.0]
    for lag in range(1, max_lags + 1):
        c = ((s[lag:] - m) * (s[:-lag] - m)).mean()
        acf.append(c / v if v > 0 else 0.0)
    return np.array(acf)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
conf = 1.96 / np.sqrt(len(df_merged["imbl_price"].dropna()))

for ax, col, title in [
    (axes[0], "imbl_price", "ACF: Imbalance Price"),
    (axes[1], "direction",  "ACF: Direction"),
]:
    if HAS_STATSMODELS:
        sm_acf(df_merged[col].dropna(), lags=MAX_LAGS, ax=ax, alpha=0.05)
    else:
        acf_vals = _manual_acf(df_merged[col])
        lags_x   = np.arange(len(acf_vals))
        ax.bar(lags_x, acf_vals, width=0.6, color="steelblue", alpha=0.7)
        ax.axhline(conf,  color="red", linestyle="--", linewidth=0.8)
        ax.axhline(-conf, color="red", linestyle="--", linewidth=0.8)

    ax.set_xlabel("Lag (15-min periods)")
    ax.set_title(f"{title} (up to 12 h, 48 lags)")
    ax.axvline(4,  color="orange", linewidth=0.8, linestyle=":", label="1 h")
    ax.axvline(16, color="green",  linewidth=0.8, linestyle=":", label="4 h")
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "05_autocorrelation.png", dpi=150)
plt.show()

### 2f. Wind Speed vs Imbalance Price

Wind surplus drives down spot prices; short imbalance (deficit) spikes when
wind is weaker than expected.  We also look at *wind surprise* (actual minus
7-day rolling mean) which is a forward-looking signal.

In [ ]:
wind_col = "windspeed_100m" if "windspeed_100m" in df_merged.columns else "windspeed_10m"
w = df_merged[wind_col].dropna()
p_aligned = df_merged["imbl_price"].reindex(w.index)
valid = w.notna() & p_aligned.notna()
w_v, p_v = w[valid], p_aligned[valid]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter with binned means
axes[0].scatter(w_v, p_v, alpha=0.03, s=2, color="steelblue", rasterized=True)
bins = pd.cut(w_v, bins=20)
bin_means = p_v.groupby(bins).mean()
bin_centers = [iv.mid for iv in bin_means.index]
axes[0].plot(bin_centers, bin_means.values, color="red", linewidth=2,
             label="Binned mean")
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel(f"{wind_col} (m/s)")
axes[0].set_ylabel("Imbalance price (EUR/MWh)")
axes[0].set_title("Wind speed vs imbalance price")
axes[0].legend()

# Wind surprise vs imbalance price
w_7d_mean = w.rolling(672, min_periods=48).mean()
w_surprise = (w - w_7d_mean).reindex(p_v.index)
vs_valid = w_surprise.notna() & p_v.notna()
ws, pv2 = w_surprise[vs_valid], p_v[vs_valid]

axes[1].scatter(ws, pv2, alpha=0.03, s=2, color="darkorange", rasterized=True)
bins2 = pd.cut(ws, bins=20)
bm2 = pv2.groupby(bins2).mean()
bc2 = [iv.mid for iv in bm2.index]
axes[1].plot(bc2, bm2.values, color="red", linewidth=2, label="Binned mean")
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Wind surprise (actual − 7-day rolling mean, m/s)")
axes[1].set_ylabel("Imbalance price (EUR/MWh)")
axes[1].set_title("Wind surprise vs imbalance price")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_wind_vs_price.png", dpi=150)
plt.show()

In [ ]:
p = df_merged["imbl_price"].dropna()
print("=" * 42)
print("Key statistics — SE3 imbalance price")
print("=" * 42)
print(f"  Count   : {len(p):>10,}")
print(f"  Mean    : {p.mean():>10.2f}  EUR/MWh")
print(f"  Std     : {p.std():>10.2f}  EUR/MWh")
print(f"  Min     : {p.min():>10.2f}  EUR/MWh")
print(f"  Max     : {p.max():>10.2f}  EUR/MWh")
print(f"  % < 0   : {(p < 0).mean()*100:>9.1f}%")
print(f"  % > 200 : {(p > 200).mean()*100:>9.1f}%")
print("=" * 42)

## 3. Feature Engineering

Adapted from `ml.py`'s `build_features()` for 15-min imbalance data.
Key differences from the day-ahead model:

| Aspect | Day-ahead (ml.py) | Imbalance (this notebook) |
|---|---|---|
| Lags | ≥ 24 h | 15 min – 4 h (short lags) |
| Direction | N/A | direction lags + streak |
| Fourier | hourly 24-slot | 15-min 96-slot |
| Ridge baseline | yes (trivial features) | no (imbalance lacks clean linear structure) |


**Why short lags instead of 24 h+ as in `ml.py`?**

Day-ahead spot prices are published the evening before, so the minimum
information lag is 24 h. Imbalance prices are determined in real-time, so
the last known period is only 15 minutes old. The autocorrelation plot
(Section 2e) confirms that the lag-1 through lag-16 periods carry far more
signal than anything beyond a few hours.

**Leakage policy:**
All price-derived features are shifted so that at prediction time t, the model
only sees data from ≤ t − 1 period. Weather features are *not* shifted because
they represent observed (not future) conditions and are conceptually available
at prediction time — consistent with how `ml.py` treats weather inputs.

**Guard pattern used throughout:**
```python
if col in X.columns and X[col].notna().any():
    X[derived_col] = ...   # only create if source has real data
```
This prevents all-NaN derived columns that would later cause `dropna` to
silently empty the training or test set.

In [ ]:
def _direction_streak(d: pd.Series) -> pd.Series:
    """
    Count consecutive periods in the same direction (vectorised, lagged 1).

    Uses lagged direction to avoid leakage: the current period's direction
    is the target-adjacent signal, so we only look at t-1 and earlier.
    """
    d_lag = d.shift(1)
    changed = (d_lag != d_lag.shift(1)).fillna(True)
    grp = changed.cumsum()
    return d_lag.groupby(grp).cumcount().add(1).astype(float)


def build_imbalance_features(
    df: pd.DataFrame,
    horizon: int = HORIZON,
) -> pd.DataFrame:
    """
    Build feature matrix for SE3 imbalance price forecasting.

    All price-derived features are lagged to avoid leakage.
    Weather features are at time t (observed, not forecasted — valid since
    we are doing very-short-horizon nowcasting).

    Parameters
    ----------
    df : pd.DataFrame
        Merged 15-min DataFrame with columns: imbl_price, direction,
        windspeed_100m (or windspeed_10m), temperature, cloudcover, etc.
    horizon : int
        Forecast horizon in 15-min periods.  Default 1 (next 15 min).

    Returns
    -------
    pd.DataFrame
        Feature matrix including target column 'target'.
    """
    X = df.copy()
    X.index = pd.to_datetime(X.index)
    if X.index.tz is None:
        X.index = X.index.tz_localize(
            "Europe/Stockholm", ambiguous="infer", nonexistent="shift_forward"
        )

    p = X["imbl_price"]
    d = X["direction"].astype(float) if "direction" in X.columns else pd.Series(0, index=X.index)

    # ── Imbalance price lags ──────────────────────────────────────────────────
    for lag in [1, 2, 4, 8, 16]:
        X[f"imbl_lag_{lag}"] = p.shift(lag)
    X["imbl_lag_1h"]  = p.shift(4)
    X["imbl_lag_2h"]  = p.shift(8)
    X["imbl_lag_4h"]  = p.shift(16)

    # ── Direction lags ────────────────────────────────────────────────────────
    for lag in [1, 2, 4, 8]:
        X[f"dir_lag_{lag}"] = d.shift(lag)
    X["dir_streak"] = _direction_streak(d)

    # ── Rolling statistics on imbalance price (shift(1) to avoid leakage) ────
    p_lag = p.shift(1)
    X["imbl_roll_1h_mean"]  = p_lag.rolling(4).mean()
    X["imbl_roll_1h_std"]   = p_lag.rolling(4).std()
    X["imbl_roll_4h_mean"]  = p_lag.rolling(16).mean()
    X["imbl_roll_4h_std"]   = p_lag.rolling(16).std()
    X["imbl_roll_1d_mean"]  = p_lag.rolling(96).mean()
    X["imbl_ewa"]           = p_lag.ewm(span=8, adjust=False).mean()

    # ── Regulation spread (lag 1) ─────────────────────────────────────────────
    if ("up_reg_price" in X.columns and "down_reg_price" in X.columns
            and X["up_reg_price"].notna().any() and X["down_reg_price"].notna().any()):
        X["reg_spread"] = (X["up_reg_price"] - X["down_reg_price"]).shift(1)
    if "imbl_spot_diff" in X.columns and X["imbl_spot_diff"].notna().any():
        X["imbl_spot_diff_lag1"] = X["imbl_spot_diff"].shift(1)
    if "spot_price" in X.columns and X["spot_price"].notna().any():
        X["spot_price_lag1"] = X["spot_price"].shift(1)

    # ── Calendar ──────────────────────────────────────────────────────────────
    X["hour"]      = X.index.hour
    X["minute"]    = X.index.minute
    X["dayofweek"] = X.index.dayofweek
    X["month"]     = X.index.month
    X["slot"]      = X["hour"] * 4 + X["minute"] // 15  # 0..95 per day

    X["hour_sin"]  = np.sin(2 * np.pi * X["hour"]      / 24)
    X["hour_cos"]  = np.cos(2 * np.pi * X["hour"]      / 24)
    X["dow_sin"]   = np.sin(2 * np.pi * X["dayofweek"] /  7)
    X["dow_cos"]   = np.cos(2 * np.pi * X["dayofweek"] /  7)
    X["month_sin"] = np.sin(2 * np.pi * X["month"]     / 12)
    X["month_cos"] = np.cos(2 * np.pi * X["month"]     / 12)

    # 15-min slot Fourier (96 slots per day)
    X["slot_sin"] = np.sin(2 * np.pi * X["slot"] / 96)
    X["slot_cos"] = np.cos(2 * np.pi * X["slot"] / 96)

    # Weekly and annual Fourier harmonics
    how = X["dayofweek"] * 96 + X["slot"]
    hoy = (X.index.dayofyear - 1) * 96 + X["slot"]
    for k in [1, 2, 3]:
        X[f"daily_sin_k{k}"]  = np.sin(2 * np.pi * k * X["slot"] / 96)
        X[f"daily_cos_k{k}"]  = np.cos(2 * np.pi * k * X["slot"] / 96)
        X[f"weekly_sin_k{k}"] = np.sin(2 * np.pi * k * how / (7 * 96))
        X[f"weekly_cos_k{k}"] = np.cos(2 * np.pi * k * how / (7 * 96))
    for k in [1, 2]:
        X[f"annual_sin_k{k}"] = np.sin(2 * np.pi * k * hoy / (365 * 96))
        X[f"annual_cos_k{k}"] = np.cos(2 * np.pi * k * hoy / (365 * 96))

    X["season"]     = X["month"].map({12:1,1:1,2:1,3:2,4:2,5:2,6:3,7:3,8:3,9:4,10:4,11:4})
    X["is_weekend"] = (X.index.dayofweek >= 5).astype(int)
    X["is_holiday"] = X.index.normalize().map(lambda d: int(d in SE_HOLIDAYS))
    X["is_peak"]    = (
        X["hour"].isin(range(7, 10)) | X["hour"].isin(range(17, 21))
    ).astype(int)
    X["is_night"]   = X["hour"].isin([23, 0, 1, 2, 3, 4, 5]).astype(int)

    # ── Weather features ──────────────────────────────────────────────────────
    wind_col = "windspeed_100m" if "windspeed_100m" in X.columns else "windspeed_10m"
    if wind_col in X.columns:
        w = X[wind_col]
        # 7-day rolling mean = 7 days × 96 slots
        X["wind_7d_mean"]  = w.rolling(672, min_periods=48).mean()
        X["wind_surprise"] = w - X["wind_7d_mean"]
        X["wind_x_night"]  = w * X["is_night"]
        X["wind_x_peak"]   = w * X["is_peak"]
        X["wind_squared"]  = w ** 2

    if "temperature" in X.columns:
        X["heating_degree"] = (15 - X["temperature"]).clip(lower=0)
        X["temp_x_peak"]    = X["temperature"] * X["is_peak"]

    if "cloudcover" in X.columns:
        X["cloudcover_lag1"] = X["cloudcover"].shift(1)

    # ── Target ────────────────────────────────────────────────────────────────
    X["target"] = p.shift(-horizon)

    return X


def make_imbalance_feature_cols(df: pd.DataFrame) -> list[str]:
    """
    Return ordered feature list for the imbalance model.
    Excludes target, raw price, and non-feature columns.

    Returns
    -------
    list[str]
        Feature columns present in `df`, deduplicated and in a logical order.
    """
    imbl_lags  = [f"imbl_lag_{l}" for l in [1, 2, 4, 8, 16]] + [
        "imbl_lag_1h", "imbl_lag_2h", "imbl_lag_4h",
    ]
    roll_feats = [
        "imbl_roll_1h_mean", "imbl_roll_1h_std",
        "imbl_roll_4h_mean", "imbl_roll_4h_std",
        "imbl_roll_1d_mean", "imbl_ewa",
    ]
    dir_feats  = [f"dir_lag_{l}" for l in [1, 2, 4, 8]] + ["dir_streak"]
    cal_feats  = [
        "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
        "slot_sin", "slot_cos",
        "season", "is_night", "is_peak", "is_weekend", "is_holiday",
    ]
    cal_feats += [f"daily_{t}_k{k}"  for k in [1, 2, 3] for t in ["sin", "cos"]]
    cal_feats += [f"weekly_{t}_k{k}" for k in [1, 2, 3] for t in ["sin", "cos"]]
    cal_feats += [f"annual_{t}_k{k}" for k in [1, 2]    for t in ["sin", "cos"]]

    weather_feats = [c for c in [
        "windspeed_100m", "windspeed_10m", "wind_7d_mean", "wind_surprise",
        "wind_x_night", "wind_x_peak", "wind_squared",
        "temperature", "heating_degree", "temp_x_peak",
        "cloudcover", "cloudcover_lag1",
    ] if c in df.columns and df[c].notna().any()]

    # Exclude columns that are present but entirely NaN
    # (e.g. spot_price_lag1 when no ENTSOE key is available)
    spot_feats = [c for c in [
        "imbl_spot_diff_lag1", "reg_spread", "spot_price_lag1",
    ] if c in df.columns and df[c].notna().any()]

    all_feats = imbl_lags + roll_feats + dir_feats + cal_feats + weather_feats + spot_feats
    seen: set[str] = set()
    result: list[str] = []
    for c in all_feats:
        if c in df.columns and c not in seen:
            seen.add(c)
            result.append(c)
    return result


In [ ]:
df_feats = build_imbalance_features(df_merged, horizon=HORIZON)

feature_cols = make_imbalance_feature_cols(df_feats)
print(f"Feature matrix: {len(df_feats):,} rows × {len(feature_cols)} features")
print(f"Target NaN (last {HORIZON} rows): {df_feats['target'].isnull().sum()}")
print(f"\nFirst 10 features: {feature_cols[:10]}")
print(f"Last  10 features: {feature_cols[-10:]}")
df_feats[feature_cols + ["target"]].describe().round(2)

## 4. Baselines

Before training, we establish simple baselines on the test set to provide
skill-score reference points.

- **Persistence**: predict t as t−1 (naive lag)
- **Rolling 1h mean**: mean of last 4 periods
- **Spot proxy**: spot price (if available, otherwise skip)


**Three-step NaN safety net applied before splitting:**

1. `df_feats.dropna(axis=1, how="all")` — removes columns that are entirely NaN
   (e.g. `spot_price_lag1` when no ENTSOE key is set).
2. `make_imbalance_feature_cols(df_feats)` — rebuilds the feature list, which
   already filters to `df[c].notna().any()`.
3. `dropna(subset=feature_cols + ["target"])` — removes only rows where a
   *surviving* feature or the target is NaN.

Without step 1–2, a single all-NaN column in `feature_cols` would cause
`dropna(subset=...)` to silently discard every row in the dataset.

In [ ]:
# Step 1: drop all-NaN columns as safety net, then rebuild feature list
df_feats      = df_feats.dropna(axis=1, how="all")
feature_cols  = make_imbalance_feature_cols(df_feats)

split_date = df_feats.index.max() - pd.Timedelta(days=TEST_DAYS)
test_mask  = df_feats.index > split_date

# Step 2: drop rows with any NaN in the surviving feature columns
df_train = df_feats[~test_mask].dropna(subset=feature_cols + ["target"])
df_test  = df_feats[ test_mask].dropna(subset=feature_cols + ["target"])

X_train, y_train = df_train[feature_cols], df_train["target"]
X_test,  y_test  = df_test[feature_cols],  df_test["target"]

print(f"Train: {len(df_train):,} rows  {df_train.index.min().date()} → {df_train.index.max().date()}")
print(f"Test : {len(df_test):,} rows  {df_test.index.min().date()}  → {df_test.index.max().date()}")

# Diagnostics — catch silent failures before they reach sklearn
print(f"\ndf_feats shape : {df_feats.shape}")
print(f"Test rows      : {test_mask.sum():,}")
all_nan_feats = [c for c in feature_cols if df_feats[c].isna().all()]
print(f"All-NaN feats  : {all_nan_feats or 'none'}")


def direction_accuracy(y_true: pd.Series, y_pred: np.ndarray,
                        current: pd.Series) -> float:
    """Fraction of periods where predicted direction matches actual direction."""
    pred_dir   = np.sign(y_pred   - current.values)
    actual_dir = np.sign(y_true.values - current.values)
    valid = (pred_dir != 0) & (actual_dir != 0)
    return float((pred_dir[valid] == actual_dir[valid]).mean()) if valid.any() else float("nan")


# current known price = imbl_lag_1 (most recent known period)
current_test = df_test["imbl_lag_1"]

# Persistence
pers_pred = df_test["imbl_lag_1"].values
pers_mae  = mean_absolute_error(y_test, pers_pred)
pers_rmse = np.sqrt(mean_squared_error(y_test, pers_pred))
pers_dir  = direction_accuracy(y_test, pers_pred, current_test)

# Rolling 1h mean
roll_pred = df_test["imbl_roll_1h_mean"].values
roll_mae  = mean_absolute_error(y_test, roll_pred)
roll_rmse = np.sqrt(mean_squared_error(y_test, roll_pred))
roll_dir  = direction_accuracy(y_test, roll_pred, current_test)

print("\n" + "=" * 54)
print(f"{'Baseline':<20} {'MAE':>8} {'RMSE':>8} {'Dir Acc':>8}")
print("-" * 54)
print(f"{'Persistence':<20} {pers_mae:>8.2f} {pers_rmse:>8.2f} {pers_dir:>8.3f}")
print(f"{'Rolling 1h mean':<20} {roll_mae:>8.2f} {roll_rmse:>8.2f} {roll_dir:>8.3f}")

if "spot_price_lag1" in df_test.columns and df_test["spot_price_lag1"].notna().any():
    spot_pred = df_test["spot_price_lag1"].values
    spot_mask = ~np.isnan(spot_pred)
    if spot_mask.sum() > 100:
        spot_mae  = mean_absolute_error(y_test[spot_mask], spot_pred[spot_mask])
        spot_rmse = np.sqrt(mean_squared_error(y_test[spot_mask], spot_pred[spot_mask]))
        spot_dir  = direction_accuracy(
            y_test[spot_mask],
            spot_pred[spot_mask],
            current_test[spot_mask],
        )
        print(f"{'Spot proxy':<20} {spot_mae:>8.2f} {spot_rmse:>8.2f} {spot_dir:>8.3f}")
print("=" * 54)

## 5. Train

Three LightGBM quantile models with early stopping.  Same architecture as
`ml.py` but without the Ridge linear baseline subtraction — imbalance prices
lack the clean linear day-of-year structure that makes it worthwhile for
day-ahead spot.

- 90 / 10 train / validation split within training data
- Early stopping patience = 50 trees
- Test set = last 60 days (denser than 90-day spot set since we have 15-min data)


**Train / test split design:**

| Parameter | Value | Rationale |
|---|---|---|
| Test set | Last 60 days | 15-min resolution gives ~5 760 test points (vs 2 160 for the 90-day hourly spot model) |
| Validation | Last 10 % of train | Early stopping reference; chronological split avoids future leakage |
| No Ridge baseline | — | Imbalance has no clean linear seasonal structure; residualisation would remove real signal |
| Quantile targets | p05 / p50 / p95 | Directly models uncertainty; coverage evaluated against the 90 % nominal level |

In [ ]:
val_n = int(len(X_train) * 0.1)
X_tr, X_val = X_train.iloc[:-val_n], X_train.iloc[-val_n:]
y_tr, y_val = y_train.iloc[:-val_n], y_train.iloc[-val_n:]

cbs = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]

models: dict[str, lgb.LGBMRegressor] = {}
for alpha, name in [(0.05, "p05"), (0.50, "p50"), (0.95, "p95")]:
    print(f"Training LightGBM {name} (quantile={alpha})...")
    m = lgb.LGBMRegressor(objective="quantile", alpha=alpha, **LGBM_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=cbs)
    models[name] = m
    print(f"  → best iteration: {m.best_iteration_}\n")

print("Training complete.")

## 6. Evaluate

### 6a. Metrics vs Baselines

In [ ]:
pred_p50 = models["p50"].predict(X_test)
pred_p05 = models["p05"].predict(X_test)
pred_p95 = models["p95"].predict(X_test)

lgb_mae  = mean_absolute_error(y_test, pred_p50)
lgb_rmse = np.sqrt(mean_squared_error(y_test, pred_p50))
lgb_dir  = direction_accuracy(y_test, pred_p50, current_test)
coverage = float(((y_test >= pred_p05) & (y_test <= pred_p95)).mean())

# Skill score vs persistence (1 = perfect, 0 = same as persistence)
skill_mae = 1 - lgb_mae / pers_mae

print("=" * 60)
print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8} {'DirAcc':>8} {'Skill':>8}")
print("-" * 60)
print(f"{'Persistence':<22} {pers_mae:>8.2f} {pers_rmse:>8.2f} {pers_dir:>8.3f} {'—':>8}")
print(f"{'Rolling 1h mean':<22} {roll_mae:>8.2f} {roll_rmse:>8.2f} {roll_dir:>8.3f} {'—':>8}")
print(f"{'LightGBM p50':<22} {lgb_mae:>8.2f} {lgb_rmse:>8.2f} {lgb_dir:>8.3f} {skill_mae:>8.3f}")
print("-" * 60)
print(f"  Prediction interval coverage (p05-p95): {coverage*100:.1f}%  (target ~90%)")
print("=" * 60)

### 6b. Actual vs Predicted — last 7 days of test set

In [ ]:
last7_mask = df_test.index >= df_test.index.max() - pd.Timedelta(days=7)
df_last7   = df_test[last7_mask]
y_last7    = y_test[last7_mask]
X_last7    = X_test[last7_mask]
p50_l7     = models["p50"].predict(X_last7)
p05_l7     = models["p05"].predict(X_last7)
p95_l7     = models["p95"].predict(X_last7)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: actual vs p50 + p05-p95 band
axes[0].fill_between(df_last7.index, p05_l7, p95_l7,
                      alpha=0.25, color="steelblue", label="p05-p95")
axes[0].plot(df_last7.index, y_last7.values, color="black",
             linewidth=1, label="Actual", alpha=0.8)
axes[0].plot(df_last7.index, p50_l7, color="steelblue",
             linewidth=1.5, label="p50 forecast")
axes[0].axhline(0, color="red", linewidth=0.6, linestyle="--")
axes[0].set_ylabel("Imbalance price (EUR/MWh)")
axes[0].set_title("Actual vs Forecast — last 7 days of test set")
axes[0].legend(loc="upper right", fontsize=9)

# Bottom: residuals bar chart
residuals = y_last7.values - p50_l7
colors = ["#ef4444" if r > 0 else "#3b82f6" for r in residuals]
axes[1].bar(df_last7.index, residuals, color=colors, width=pd.Timedelta(minutes=12),
            alpha=0.7, label="Residual (actual − p50)")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Residual (EUR/MWh)")
axes[1].set_title("Residuals")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_actual_vs_forecast.png", dpi=150)
plt.show()
print(f"Last-7d MAE: {mean_absolute_error(y_last7, p50_l7):.2f} EUR/MWh")

### 6c. Feature Importance (top 30, colour-coded by group)

In [ ]:
imp   = pd.Series(models["p50"].feature_importances_, index=feature_cols)
top30 = imp.nlargest(30).sort_values()

# Colour-code by feature group
def _group_color(name: str) -> str:
    if name.startswith("imbl_lag") or name.startswith("imbl_roll") or name == "imbl_ewa":
        return "#3b82f6"   # blue = imbalance lags
    if name.startswith("dir_"):
        return "#8b5cf6"   # purple = direction
    if any(name.startswith(p) for p in ["wind", "temperature", "cloudcover",
                                          "heating", "temp_x"]):
        return "#22c55e"   # green = weather
    if any(name.startswith(p) for p in ["spot", "reg_spread", "imbl_spot"]):
        return "#f97316"   # orange = spot / regulation
    return "#9ca3af"       # grey = calendar

colors = [_group_color(n) for n in top30.index]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top30.index, top30.values, color=colors, alpha=0.85)
ax.set_xlabel("Feature importance (LightGBM split gain)")
ax.set_title("Top 30 features — LightGBM p50 model")

# Legend patches
import matplotlib.patches as mpatches
legend_items = [
    mpatches.Patch(color="#3b82f6", label="Imbalance lags"),
    mpatches.Patch(color="#8b5cf6", label="Direction"),
    mpatches.Patch(color="#22c55e", label="Weather"),
    mpatches.Patch(color="#f97316", label="Spot / regulation"),
    mpatches.Patch(color="#9ca3af", label="Calendar"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "08_feature_importance.png", dpi=150)
plt.show()

### 6d. Multi-horizon Comparison

Train and evaluate for horizons 1, 4, 8 periods (15 min, 1 h, 2 h).
Shows MAE, skill score vs persistence, and direction accuracy.

In [ ]:
HORIZONS = [1, 4, 8]   # 15 min, 1 h, 2 h
horizon_results: dict[int, dict] = {}

for h in HORIZONS:
    print(f"\n--- Horizon {h} ({h*15} min) ---")
    df_h = build_imbalance_features(df_merged, horizon=h)
    fc_h = make_imbalance_feature_cols(df_h)

    df_tr_h = df_h[~test_mask].dropna(subset=fc_h + ["target"])
    df_te_h = df_h[ test_mask].dropna(subset=fc_h + ["target"])
    X_tr_h, y_tr_h = df_tr_h[fc_h], df_tr_h["target"]
    X_te_h, y_te_h = df_te_h[fc_h], df_te_h["target"]

    val_n_h = int(len(X_tr_h) * 0.1)
    cbs_h   = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
    m_h     = lgb.LGBMRegressor(objective="quantile", alpha=0.5, **LGBM_PARAMS)
    m_h.fit(X_tr_h.iloc[:-val_n_h], y_tr_h.iloc[:-val_n_h],
            eval_set=[(X_tr_h.iloc[-val_n_h:], y_tr_h.iloc[-val_n_h:])],
            callbacks=cbs_h)

    pred_h   = m_h.predict(X_te_h)
    pers_h   = df_te_h["imbl_lag_1"].values  # persistence is always lag_1
    cur_h    = df_te_h["imbl_lag_1"]
    mae_h    = mean_absolute_error(y_te_h, pred_h)
    mae_p_h  = mean_absolute_error(y_te_h, pers_h)
    skill_h  = 1 - mae_h / mae_p_h
    dir_h    = direction_accuracy(y_te_h, pred_h, cur_h)

    horizon_results[h] = {
        "mae": mae_h, "persistence_mae": mae_p_h,
        "skill": skill_h, "dir_acc": dir_h,
        "best_iter": m_h.best_iteration_,
    }
    print(f"  MAE={mae_h:.2f}  Skill={skill_h:.3f}  DirAcc={dir_h:.3f}  "
          f"Trees={m_h.best_iteration_}")

# ── Summary bar chart ────────────────────────────────────────────────────────
labels = [f"{h*15}\nmin" for h in HORIZONS]
maes   = [horizon_results[h]["mae"]   for h in HORIZONS]
skills = [horizon_results[h]["skill"] for h in HORIZONS]
dirs   = [horizon_results[h]["dir_acc"] for h in HORIZONS]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
x = np.arange(len(HORIZONS))
w = 0.5

axes[0].bar(x, maes, width=w, color="steelblue", alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("MAE (EUR/MWh)"); axes[0].set_title("MAE by horizon")

axes[1].bar(x, skills, width=w, color="seagreen", alpha=0.8)
axes[1].axhline(0, color="red", linewidth=0.8, linestyle="--")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Skill score vs persistence"); axes[1].set_title("Skill by horizon")

axes[2].bar(x, dirs, width=w, color="darkorange", alpha=0.8)
axes[2].axhline(0.5, color="red", linewidth=0.8, linestyle="--", label="Random")
axes[2].set_xticks(x); axes[2].set_xticklabels(labels)
axes[2].set_ylabel("Direction accuracy"); axes[2].set_title("Direction accuracy by horizon")
axes[2].legend(fontsize=8)

fig.suptitle("Multi-horizon performance (LightGBM p50)", fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_multi_horizon.png", dpi=150)
plt.show()

## 7. Summary

Key results and pointers for next steps.

In [ ]:
print("=" * 62)
print("SE3 IMBALANCE PRICE FORECASTING — SUMMARY")
print("=" * 62)
print(f"  Data period  : {df_imbl.index.min().date()} → {df_imbl.index.max().date()}")
print(f"  Resolution   : 15-min  ({len(df_imbl):,} periods)")
print(f"  Features     : {len(feature_cols)}")
print(f"  Test set     : last {TEST_DAYS} days  ({len(df_test):,} periods)")
print()
print(f"  Baseline (persistence) MAE : {pers_mae:.2f}  EUR/MWh")
print(f"  LightGBM p50 MAE           : {lgb_mae:.2f}  EUR/MWh")
print(f"  Skill score vs persistence : {skill_mae:.3f}")
print(f"  Direction accuracy         : {lgb_dir:.3f}")
print(f"  PI coverage (p05-p95)      : {coverage*100:.1f}%  (target ~90%)")
print()
print("  Multi-horizon MAE:")
for h in HORIZONS:
    r = horizon_results[h]
    print(f"    {h*15:>4} min  MAE={r['mae']:.2f}  Skill={r['skill']:.3f}  DirAcc={r['dir_acc']:.3f}")
print()
print("  Saved plots:")
for f in sorted(FIGURES_DIR.glob("*.png")):
    print(f"    {f}")
print("=" * 62)
print()
print("NEXT STEPS:")
print("  1. Wrap data fetch + feature build in tools/imbalance.py")
print("  2. Extend pipeline.py with sync_imbalance() → imbalance table in DuckDB")
print("  3. Wire into graph.py as a new agent tool for live imbalance forecasting")
print("  4. Add SMHI wind forecast as a forward-looking feature (see pipeline.py)")
print("  5. Consider XGBoost or CatBoost as ensemble components")